# ليلة وناي وربابة — مونتاج 50 ثانية

هذه النسخة تعمل بالكامل على **Google Colab**.  
خامات الفيديو والأغنية تُحمَّل إلى سيرفر Google مباشرة، وليس إلى الهاتف.

1. شغّل خلية **BUILD** مرة واحدة.
2. شاهد المعاينة داخل Colab.
3. شغّل خلية **DOWNLOAD PREVIEW** لتنزيل النسخة الخفيفة فقط.
4. لو أعجبتك، شغّل **DOWNLOAD FULL** لتنزيل نسخة 720×1280.

المونتاج: ليل/مطر → مدينة → صحاب/دفء → فجر/شروق.


In [ ]:
# BUILD — شغّل هذه الخلية مرة واحدة
import os, subprocess, requests
from pathlib import Path
from IPython.display import Video, display

ROOT = Path('/content/reel_montage')
SRC = ROOT / 'src'
CLIPS = ROOT / 'clips'
OUT = ROOT / 'output'
for p in (SRC, CLIPS, OUT):
    p.mkdir(parents=True, exist_ok=True)

subprocess.run('ffmpeg -version >/dev/null 2>&1 || (apt-get -qq update && apt-get -qq install -y ffmpeg)', shell=True, check=True)

sources = {
    'song.mp3': 'https://cdn.creativeclaw.co/u/2eb76212/audio/974df573-c163-4d17-81c1-94d92a4b4265.mp3',
    'rain_traffic.mp4': 'https://cdn.creativeclaw.co/u/2eb76212/videos/fa51da72-d5cc-4978-be0f-a533733d6109.mp4',
    'rain_window.mp4': 'https://cdn.creativeclaw.co/u/2eb76212/videos/ca11d0c5-8b6a-4563-8772-7dd8501c394e.mp4',
    'vertical_traffic.mp4': 'https://cdn.creativeclaw.co/u/2eb76212/videos/24120068-d357-4d5c-90f2-2e6104fdc688.mp4',
    'friends_cafe.mp4': 'https://cdn.creativeclaw.co/u/2eb76212/videos/53bd9e96-7b18-433b-b33a-a1c78f13e2af.mp4',
    'sunrise_city.mp4': 'https://cdn.creativeclaw.co/u/2eb76212/videos/904668b0-3061-41e8-ba06-784fd7cdb0c0.mp4',
    'sunrise_road.mp4': 'https://cdn.creativeclaw.co/u/2eb76212/videos/df967382-e534-4509-9772-3b27c25d81be.mp4',
}

def download(name, url):
    dest = SRC / name
    if dest.exists() and dest.stat().st_size > 100000:
        print('✓ موجود:', name)
        return
    print('↓ تحميل للسيرفر:', name)
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(dest, 'wb') as f:
            for chunk in r.iter_content(chunk_size=1024*1024):
                if chunk:
                    f.write(chunk)
    print('  ', round(dest.stat().st_size/1024/1024, 1), 'MB')

for name, url in sources.items():
    download(name, url)

CLIP = 3.90244
plan = [
    ('rain_window.mp4',       0.0, 1.08, 0.78, -0.045, 1.10),
    ('rain_traffic.mp4',      1.0, 0.94, 0.82, -0.040, 1.12),
    ('vertical_traffic.mp4',  0.0, 0.90, 0.88, -0.035, 1.10),
    ('rain_window.mp4',       3.0, 1.00, 0.80, -0.040, 1.11),
    ('rain_traffic.mp4',      4.0, 0.88, 0.86, -0.030, 1.10),
    ('friends_cafe.mp4',      0.0, 1.02, 0.96, -0.005, 1.05),
    ('friends_cafe.mp4',      3.5, 0.94, 1.02,  0.000, 1.05),
    ('vertical_traffic.mp4',  4.0, 0.90, 0.94, -0.020, 1.08),
    ('sunrise_city.mp4',      0.0, 1.05, 1.02,  0.000, 1.06),
    ('sunrise_road.mp4',      0.0, 1.00, 1.04,  0.005, 1.05),
    ('sunrise_city.mp4',      3.5, 0.96, 1.06,  0.008, 1.05),
    ('sunrise_road.mp4',      3.0, 0.92, 1.08,  0.010, 1.05),
    ('sunrise_road.mp4',      6.0, 1.06, 1.10,  0.012, 1.04),
]

def run(cmd):
    subprocess.run(cmd, shell=True, check=True)

for i, (src, start, speed, sat, bright, contrast) in enumerate(plan, 1):
    out = CLIPS / f'{i:02d}.mp4'
    vf = (
        f'scale=720:1280:force_original_aspect_ratio=increase,'
        f'crop=720:1280,fps=30,setsar=1,'
        f'setpts={speed}*PTS,'
        f'eq=saturation={sat}:brightness={bright}:contrast={contrast},'
        f'trim=duration={CLIP},setpts=PTS-STARTPTS'
    )
    cmd = (
        f"ffmpeg -y -hide_banner -loglevel error -stream_loop -1 -ss {start} "
        f"-i '{SRC/src}' -t 6 -an -vf \"{vf}\" "
        f"-c:v libx264 -preset veryfast -crf 21 -pix_fmt yuv420p '{out}'"
    )
    run(cmd)

concat_txt = ROOT / 'concat.txt'
with open(concat_txt, 'w') as f:
    for p in sorted(CLIPS.glob('*.mp4')):
        f.write(f"file '{p}'\n")

run(f"ffmpeg -y -hide_banner -loglevel error -f concat -safe 0 -i '{concat_txt}' -an -c copy '{ROOT/'video_concat.mp4'}'")
run(f"ffmpeg -y -hide_banner -loglevel error -ss 252.8 -t 50 -i '{SRC/'song.mp3'}' -af \"afade=t=in:st=0:d=0.08,afade=t=out:st=49.25:d=0.75\" -c:a aac -b:a 192k '{ROOT/'audio_50.m4a'}'")

FULL = OUT / 'leila_nay_rababa_reel_50s_720x1280.mp4'
PREVIEW = OUT / 'leila_nay_rababa_reel_50s_PREVIEW_540x960.mp4'
run(f"ffmpeg -y -hide_banner -loglevel error -i '{ROOT/'video_concat.mp4'}' -i '{ROOT/'audio_50.m4a'}' -t 50 -map 0:v:0 -map 1:a:0 -c:v libx264 -preset veryfast -b:v 2800k -maxrate 3200k -bufsize 6400k -c:a aac -b:a 160k -pix_fmt yuv420p -movflags +faststart '{FULL}'")
run(f"ffmpeg -y -hide_banner -loglevel error -i '{FULL}' -vf scale=540:960 -c:v libx264 -preset veryfast -b:v 1050k -maxrate 1250k -bufsize 2500k -c:a aac -b:a 128k -pix_fmt yuv420p -movflags +faststart '{PREVIEW}'")

print('\n✅ انتهى المونتاج')
print('Preview:', round(PREVIEW.stat().st_size/1024/1024, 1), 'MB')
print('Full   :', round(FULL.stat().st_size/1024/1024, 1), 'MB')
display(Video(str(PREVIEW), embed=False, width=360))


In [ ]:
# DOWNLOAD PREVIEW — شغّلها لو عايز النسخة الخفيفة فقط
from google.colab import files
files.download('/content/reel_montage/output/leila_nay_rababa_reel_50s_PREVIEW_540x960.mp4')


In [ ]:
# DOWNLOAD FULL — اختياري بعد ما تعجبك المعاينة
from google.colab import files
files.download('/content/reel_montage/output/leila_nay_rababa_reel_50s_720x1280.mp4')
